# ParFlow-CONUS1 Streamflow Evaluation  
#### _Use HydroData to Retrieve Modeled and Observed Streamflow Data for a Watershed of Interest with ParFlow-CONUS Outputs vs USGS Observed Streamflow - Full Evaluation Workflow_
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org), Danielle Tijerina-Kreuzer (dtijerina@cuahsi.org), Amy Defnet (amydefnet@princeton.edu)  
Last updated: March 2026

### Introduction:  
This notebook demonstrates how to perform a point-scale analysis comparing modeled and observed streamflow at selected USGS stream gages. We focus on analyzing model performance for **site-by-site streamflow comparisons**.   

## 1. Setup

### 1a. Python Environment  

Ensure that the `cssi_evaluation` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Set up the recommended local environment" in the [GettingStarted.md](https://github.com/hydroframe/cssi_evaluation/blob/main/GettingStarted.md) file.

Import the libraries needed to run this notebook:

In [ ]:
import datetime
from pathlib import Path
import pandas as pd
import numpy as np
import hf_hydrodata as hf
import parflow as pf
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from cssi_evaluation.utils import plot_utils
from cssi_evaluation.utils import evaluation_utils
from cssi_evaluation.variables import streamflow_utils

%load_ext autoreload
%autoreload 2


### 1b. Register Pin and Access HydroData

To access the HydroData catalog you will need to sign up for a [HydroFrame account](https://hydrogen.princeton.edu/signup) (do this only once), [create a 4-digit PIN](https://hydrogen.princeton.edu/pin), and register your PIN in order to have access to the HydroData datasets (you will do this in the next code cell below). To note, you PIN will expire after 7 days and will need to recreate it after that time. The [Getting Started page](https://hf-hydrodata.readthedocs.io/en/latest/getting_started.html) has full details on how to sign up for an account and set up a PIN.

In [ ]:
# Once you create a PIN, you'll need to register your PIN.
hf.register_api_pin("your_email", "your_api_pin")

## 2. Define geographic domain and date range

### 2a. Define date range and (optional) file directories

In [ ]:
# Start and end times of a water year (to note, these dates were chosen to align with the PFCONUS1 early 2000s runs)
StartDate = '2002-10-01'
EndDate = '2003-09-30'

domain_data_path = 'examples/parflow/domain_data/' # path to the model domain data

# Output folders for modeled and observed data
OBS_OutputFolder = Path("./obs_outputs")
MOD_OutputFolder = Path("./mod_outputs")

OBS_OutputFolder.mkdir(parents=True, exist_ok=True)
MOD_OutputFolder.mkdir(parents=True, exist_ok=True)

### 2b. Define the watershed of interest

One of the simplest ways to gather data and model output from HydroData is by specifying a [Hydrologic Unit Code](https://www.usgs.gov/national-hydrography/watershed-boundary-dataset). Before we retrieve any hydrologic information, we need to indicate a HUC8 code and use it to gather streamflow observations from USGS stream gages.  

✏️ If you have a specific HUC in mind, you can change the variable `huc_code` below.

In [ ]:
# ✏️ Specify HUC8 ID and Name for watershed of interest
# (The Bayou De Chien HUC '08010201' has only 1 gage, which will run faster than the following Patoka-White HUC '051202')
huc_code = '051202'
print(f'HUC ID: {huc_code}')

huc_name = 'Patoka-White'
print(f'HUC name: {huc_name}')

## 3. Retrieve Observed USGS Streamflow Data via `hf_hydrodata`

Here we are providing the code needed to access streamflow observations data from the `hf_hydrodata` Python package for our HUC. If you'd like more detail on this code and the `hf_hydrodata` package, please see the notebook `examples/collect_observations/hydrodata_collect_observations.ipynb`.  

##### `metadata_df`
One of the dataframes we collect is the metadata dataframe. This is an important addition to the observations and it is recommended to always gather and save this for the observations you are using (particularly to support reproducibility within an open-science workflow). The saved file has useful attributes like site names, first and last date of available data, lat/lon, and the query URL.  

Additionally, the metadata contains **ParFlow-CONUS1 and ParFlow-CONUS2 `i,j` indices, which indicate the exact model domain grid cell the observation aligns with**. This is a useful HydroData feature that removes the need for users to manually match station latitude/longitude coordinates to the appropriate model grid cell, as this spatial mapping is handled directly within HydroData. We will use these indices below to extract PF-CONUS1 modeled streamflow for each USGS gage in the section below. 

In [ ]:
# Create dictionary of options to pass to the HydroData API for point-level streamflow data
point_options = {
    "dataset": "usgs_nwis",
    "variable": "streamflow",
    "temporal_resolution": "daily",
    "aggregation": "mean",
    "huc_id": [huc_code], 
    "grid": "conus1",
    "date_start": StartDate,
    "date_end": EndDate
}

# Let's request site-level metadata first. 
streamflow_metadata_df = hf.get_point_metadata(point_options)

print(f"Number of streamflow sites: {len(streamflow_metadata_df)}")
streamflow_metadata_df.head(5)

Plot the locations of the streamflow gages onto a map:

In [ ]:
# Let's define a helper plotting function to use throughout the notebook
def plot_sites_on_map(metadata_df, color_by=None):
    """
    Plots site locations on an interactive map using Plotly.

    Parameters:
    - metadata_df: DataFrame with 'latitude', 'longitude', 'site_name', 'site_id'
    - color_by: column name to color by (e.g., 'nse', 'kge', 'site_type')
    """
    pio.renderers.default = "notebook"

    center = {
        "lat": metadata_df["latitude"].mean(),
        "lon": metadata_df["longitude"].mean()
    }

    fig = px.scatter_map(
        metadata_df,
        lat="latitude",
        lon="longitude",
        color=color_by,  
        hover_name="site_name",
        hover_data=["site_id"],
        center=center,
        zoom=8,
        height=600,
        color_continuous_scale="RdYlGn",  # good default for performance metrics
    )

    fig.update_layout(mapbox_style="carto-voyager")
    fig.update_traces(marker=dict(size=12))

    return fig

plot_sites_on_map(streamflow_metadata_df)

##### Request USGS Streamflow data through the HydroData API 
Pull streamflow for each of the USGS gages located in the given HUC and time period using the `hf.get_point_data()` function. This is based on our `point_options` dictionary above. 

In [ ]:
# Now let's request data.
streamflow_data_df = hf.get_point_data(point_options)
print(f"Streamflow data shape: {streamflow_data_df.shape}")

Clean up any sites that have no data for both `streamflow_data_df` and  `streamflow_metadata_df`:

In [ ]:
# Identify columns with NO NaNs
valid_sites = streamflow_data_df.columns[streamflow_data_df.notna().all()]

# Subset the data dataframe
streamflow_data_df = streamflow_data_df[valid_sites]

# Subset the metadata dataframe
streamflow_metadata_df = streamflow_metadata_df[
    streamflow_metadata_df["site_id"].isin(valid_sites)
]

print(f"Number of valid streamflow sites: {len(valid_sites)}")
streamflow_data_df.head(5)

## 4. Retrieve ParFlow-CONUS1 Modeled Streamflow Data

<div style="color:black;background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>📖 Note</h4>
<p>
As a project, we have pre-computed the ParFlow grid cell mappings for every stream gage, groundwater well, SNOTEL station, SCAN station, and AmeriFlux site. Those are stored in the variables <code>conus1_i</code> and <code>conus1_j</code> for the ParFlow-CONUS1 grid and <code>conus2_i</code> and <code>conus2_j</code> for the ParFlow-CONUS2 grid. In this example, we will use those grid indices directly. Please see the <code>parflow_grid_mapping.ipynb</code> notebook in this same directory for more details on how we constructed those mappings.
</p>
</div>

Now that we know where our sites are located, we're going to extract only those grid cells. The ParFlow-CONUS1 simulation results are a gridded dataset, so we'll use the `get_gridded_data` function. This function takes inputs similar to the `get_point_data` function; sometimes fewer parameters are needed. We'll request from the [ParFlow-CONUS1 modern simulations](https://hf-hydrodata.readthedocs.io/en/latest/gen_conus1_baseline_mod.html) dataset.
- `dataset="conus1_baseline_mod"`
- `variable="streamflow"`
- `temporal_resolution="daily"`

**Note**: The `get_gridded_data` function follows NumPy indexing convention. The end date will be exclusive of the request. We are working on updating the point module to be consistent with this convention. For now, set the start and end dates separately within each query.

Because there are only specific locations we want to extract, we'll use the `grid_point` parameter to extract the grid cell associated with each site. As a reminder, the [Python API reference](https://hf-hydrodata.readthedocs.io/en/latest/api_reference.html) has complete information on available parameter options.

In [ ]:
date_start = datetime.datetime.strptime(StartDate, '%Y-%m-%d')
date_end = datetime.datetime.strptime(EndDate, '%Y-%m-%d')

# Create DataFrame for storing ParFlow results
timesteps = np.arange(
            date_start,
            date_end + datetime.timedelta(days=1),
            datetime.timedelta(days=1)).astype("datetime64[D]")

model_df = pd.DataFrame(columns=["date"], data=timesteps)
model_df.head(5)

### Compute simulated streamflow from PF-CONUS1 pressure outputs 

In [ ]:
# Gather static fields needed for streamflow calculation
mannings = 5.0e-5
dx, dy = 1000, 1000

slope_x = hf.get_gridded_data({
        "dataset": "conus1_domain",
        "variable": "slope_x",
    })
slope_y = hf.get_gridded_data({
        "dataset": "conus1_domain",
        "variable": "slope_y",
    })

In [ ]:
#  Loop over each station in obs_metadata_df
for idx, row in streamflow_metadata_df.iterrows():
    site_id = row["site_id"]  # original USGS site_id
    conus_i = int(row["conus1_i"])
    conus_j = int(row["conus1_j"])
    print(f"Processing site {site_id} at PF-CONUS1 grid point ({conus_i}, {conus_j})")
    
    # Build options dict for this station
    parflow_options = {
        "dataset": "conus1_baseline_mod",
        "variable": "pressure_head",
        "temporal_resolution": "daily",
        "start_time": date_start,
        "end_time": date_end + datetime.timedelta(days=1),  ### NOTE: the gridded function has exclusive end date
        "grid_point": [conus_i, conus_j]
    }
    print(f"Requesting pressure data for site {site_id}")
    pressure = hf.get_gridded_data(parflow_options)
    
    # Loop through timesteps to calculate streamflow at each timestep
    print(f"Calculating streamflow for site {site_id}")
    streamflow = np.empty(len(pressure))
    for t in range(len(pressure)):
        streamflow[t] = pf.tools.hydrology.calculate_overland_flow(pressure[t].reshape(5, 1, 1), 
                                           slope_x[conus_j, conus_i].reshape(1, 1),
                                           slope_y[conus_j, conus_i].reshape(1, 1),
                                           mannings, dx, dy, mask=np.ones((5, 1, 1)), 
                                           flow_method="OverlandFlow") / 3600  # convert from m^3/h to cms
            
    # Fill column in model_df
    model_df[site_id] = np.squeeze(np.array(streamflow))

In [ ]:
model_df.head(5)

In [ ]:
streamflow_data_df.head(5)

Great! Now we have our ParFlow results in the same format as our observations data.

*Optional*: Save observations data and matching ParFlow outputs to .csv files

In [ ]:
# ensure 'date' columns are datetime 
streamflow_data_df['date'] = pd.to_datetime(streamflow_data_df['date'])
model_df['date'] = pd.to_datetime(model_df['date'])

# Save CSVs to the output folders
streamflow_data_df.to_csv(f'{OBS_OutputFolder}/streamflow_obs_data.csv', index=False)
streamflow_metadata_df.to_csv(f'{OBS_OutputFolder}/streamflow_obs_metadata.csv', index=False)

model_df.to_csv(f'{MOD_OutputFolder}/streamflow_parflow_model_data.csv', index=False)

## 5. Evaluate ParFlow-CONUS1 outputs against observations

### 5a. Create hydrograph plots and compute metrics for each stream gage 
#### Plot the observed and modeled values together over time. This helps identify:
   - Periods of systematic bias
   - Timing differences in peaks and lows
   - General agreement in trends

In [ ]:
# The resulting plots will be saved to the current working directory.
plot_utils.plot_time_series(
    streamflow_data_df,
    model_df,
    streamflow_metadata_df,
    variable="streamflow")

>Optional: Copy the following code to a Python code block to view saved plots in the notebook.  
To note, this will display all of the saved plots, so be mindful if you have many sites.  
```from IPython.display import display, Image

for site_id in list(streamflow_metadata_df["site_id"]):
    img = f"streamflow_{site_id}.png"
    display(Image(filename=img))

In [ ]:
# look at one of the saved hydrographs in the notebook
from IPython.display import display, Image

img = "streamflow_03354000.png"
display(Image(filename=img))

#### Calculate site-level statistical metrics of comparison

In [ ]:
metrics_df = evaluation_utils.calculate_metrics(streamflow_data_df, model_df, streamflow_metadata_df)
metrics_df.head(5)

### 5b. Multi-Site Evaluation  
We begin by examining model performance across all gages in the watershed.

#### Scatter Plot for all gages with 1:1 Line:
Plots each modeled value against its corresponding observed value. This highlights:
   - Accuracy across the full range of values
   - Over- or under-prediction patterns
   - Outliers or extreme events

In [ ]:
# Scatter plot for all sites comparing values
plot_utils.plot_compare_scatter(streamflow_data_df, model_df, variable="streamflow", log_scale=False)


#### Visualize Metric Distribution  
Histograms of individual performance metrics provide a high-level evaluation of model performance across the domain.

In [ ]:
metrics_df[["nse", "kge", "spearman_rho", "bias"]].hist(bins=20)
plt.suptitle(f'Distribution of Selected Performance Metrics, n={len(metrics_df)}')

#### Spatial mapping of gage with colored by performance metric 

In [ ]:
plot_sites_on_map(metrics_df, color_by="bias")

#### Combination Metrics  
The Condon diagram combines Spearman's Rho and Absolute Relative Bias to represent the performance of streamflow magnitude and timing. 

In [ ]:
# Map Condon diagram
plot_utils.plot_condon_diagram(metrics_df, variable="streamflow")

### 5c. Single-Site Evaluation

In [ ]:
# pick a gage 
single_gage = "03354000"

gage_metrics = metrics_df[metrics_df["site_id"] == single_gage].iloc[0]
display(gage_metrics.to_frame(name="value"))

#### Plot streamflow diagnostic plots using the function `plot_streamflow_diagnostics()` in `streamflow_utils.py` to show modeled vs. observed hydrograph, flow duration curve, and Q-Q plot: 

In [ ]:
streamflow_utils.plot_streamflow_diagnostics(
    streamflow_data_df,
    model_df,
    single_gage,
    gage_metrics
)

FDC: plots streamflow against exceedance probability; compares the distribution of flows between observed and modeled data  
Q-Q: compares modeled and observed flows at the same percentiles (quantiles)